# Démo 3 : préparation des modèles, sur tout le dataset d'origine

**But** : préparer une démonstration sur **les 30 000 clients** du dataset d'origine, sans nettoyage métier, sans règle du contentieux ni périmètre, où **chaque client est noté par un modèle qui ne l'a jamais vu**.

**Le principe : cinq modèles, pas un.** Un modèle entraîné sur tout le dataset a vu tout le dataset. On reprend donc la validation croisée de l'essai du comparatif global (`lab_ML/comparatif_global/essai_jeu_de_base.ipynb`) : les clients sont répartis en 5 plis ; le modèle d'un pli est entraîné sur les 4 autres et n'a jamais vu les clients de son pli. Ensemble, les 5 modèles couvrent les 30 000 clients, chacun noté par le seul modèle qui ne l'a pas appris.

**Ce qui est repris de l'essai, à l'identique** (rien n'est réglé ici) :
- les données : le jeu de base (`data/creditcard_pret_ingestion.csv`, niveau 0 du nettoyage, celui chargé dans la base de données) ;
- les 23 variables d'origine (variante « Complet ») et leur encodage ;
- le modèle : **CatBoost**, avec la version retenue par la boucle de l'essai pour la variante complète (sortie de la section 3 de l'essai) ;
- les mêmes 5 plis (`StratifiedKFold`, 5 plis, `random_state=42`) ;
- le seuil de décision de l'essai, optimisé dans chaque pli par la règle marginale (au plus 1 bon client signalé pour 1 défaut attrapé, à la marge).

**Contrôle** : les probabilités recalculées doivent être celles enregistrées par l'essai (`data/ML/comparatif_global/jeu_de_base.csv`).

**Fichier produit** : `lab_ML/demo_ML/demo_3_modeles_plis.joblib` (les 5 modèles entraînés, le seuil de chacun, et pour chacun les identifiants des clients qu'il n'a pas vus).

**Étapes** :
1. chargement et variables ;
2. le modèle et les plis ;
3. entraînement des 5 modèles ;
4. contrôle du surapprentissage : chaque modèle sur ses clients vus et sur ses clients non vus ;
5. évaluation sur le test : les 20 % non vus de chaque modèle, puis les 30 000 clients ensemble ;
6. contrôle face à l'essai ;
7. enregistrement.

## 1. Chargement et variables

In [5]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, fbeta_score, make_scorer, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, TunedThresholdClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
dossier_demo = root_dir / "lab_ML" / "demo_ML"

original = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")   # niveau 0 du nettoyage, comme l'essai
assert original["ID"].is_unique and not original.isna().any().any()
y_train = original["dpnm"].reset_index(drop=True)
y = y_train.to_numpy()

# Les 23 variables d'origine, rangées selon leur encodage (essai, cellule « VARIABLES »)
PAY = [f"PAY_{n}" for n in range(1, 7)]
TYPES = {
    "num": ["LIMIT_BAL", "AGE"] + PAY,
    "log": [f"PAY_AMT{n}" for n in range(1, 7)],
    "log_signe": [f"BILL_AMT{n}" for n in range(1, 7)],
    "nom": ["SEX", "EDUCATION", "MARRIAGE"],
}
VARIABLES = [c for cols in TYPES.values() for c in cols]
assert sorted(VARIABLES) == sorted(c for c in original.columns if c not in ("ID", "dpnm"))
display(Markdown(f"**Jeu de base** : {len(original)} clients, {len(VARIABLES)} variables, "
                 f"{y.sum()} défauts ({y.mean() * 100:.2f} %)"))

**Jeu de base** : 30000 clients, 23 variables, 6636 défauts (22.12 %)

## 2. Le modèle et les plis

Pipeline et réglages de l'essai : encodage par type de colonne, CatBoost pondéré sur les défauts, version retenue par la boucle pour la variante complète (`depth 4 ; iterations 667 ; l2_leaf_reg 33.8 ; learning_rate 0.02`, tour 2). Seuil : `TunedThresholdClassifierCV`, qui choisit dans chaque pli, sur des plis internes, le seuil qui maximise le gain net de la règle marginale.

In [6]:
TRANSFORMATIONS = {
    "num": lambda: StandardScaler(),
    "log": lambda: make_pipeline(FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()),
    "log_signe": lambda: make_pipeline(FunctionTransformer(np.arcsinh, feature_names_out="one-to-one"), StandardScaler()),
    "nom": lambda: OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
}
preprocesseur = ColumnTransformer([(nom, TRANSFORMATIONS[nom](), colonnes) for nom, colonnes in TYPES.items()])
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
REGLAGE = {"depth": 4, "iterations": 667, "l2_leaf_reg": 33.8, "learning_rate": 0.02}   # version retenue par l'essai
modele = make_pipeline(preprocesseur, CatBoostClassifier(scale_pos_weight=scale_pos, random_state=42, verbose=0, thread_count=-1,
                                                         allow_writing_files=False, **REGLAGE))

# Mêmes plis que l'essai, et mêmes plis internes pour le seuil
plis = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(original, y_train))
cv_interne = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

BONS_CLIENTS_PAR_DEFAUT = 1   # règle marginale de l'essai : au plus 1 bon client signalé pour 1 défaut attrapé, à la marge

def gain_net(y_vrai, y_pred):
    y_vrai, y_pred = np.asarray(y_vrai), np.asarray(y_pred)
    detectes = (y_pred & (y_vrai == 1)).sum()
    fausses_alertes = (y_pred & (y_vrai == 0)).sum()
    return (detectes - BONS_CLIENTS_PAR_DEFAUT * fausses_alertes) / len(y_vrai)

SCORE_SEUIL = make_scorer(gain_net)
display(Markdown(f"**Plis** : {[len(v) for _, v in plis]} clients non vus par pli"))

**Plis** : [6000, 6000, 6000, 6000, 6000] clients non vus par pli

## 3. Entraînement des 5 modèles

Chaque modèle est entraîné sur 4 plis (80 % des clients) ; son seuil de décision est choisi **à l'intérieur de ces 80 %**, sur des plis internes (`TunedThresholdClassifierCV`) : le pli mis de côté ne sert ni à l'entraînement, ni au choix du seuil.

In [7]:
X = original[VARIABLES].reset_index(drop=True)   # même ordre de colonnes que l'essai (variante complète)
modeles, lignes = [], []
for numero, (entrainement, validation) in enumerate(plis, start=1):
    m = TunedThresholdClassifierCV(clone(modele), scoring=SCORE_SEUIL, cv=cv_interne, n_jobs=-1)
    m.fit(X.iloc[entrainement], y[entrainement])
    modeles.append(m)
    lignes.append({"Pli mis de côté": numero, "Clients d'entraînement": len(entrainement), "Clients mis de côté": len(validation),
                   "Seuil choisi (dans l'entraînement)": round(m.best_threshold_, 3)})
display(pd.DataFrame(lignes).set_index("Pli mis de côté"))

,Clients d'entraînement,Clients mis de côté,Seuil choisi (dans l'entraînement)
Pli mis de côté,,,
1,24000,6000,0.789
2,24000,6000,0.751
3,24000,6000,0.788
4,24000,6000,0.759
5,24000,6000,0.767


**Comment lire** : 5 modèles, chacun entraîné sur 24 000 clients et laissant 6 000 clients de côté ; des seuils proches d'un modèle à l'autre veulent dire que la règle de seuil se comporte de la même façon sur chaque partie des données.

## 4. Contrôle du surapprentissage

Pour chaque modèle, la même mesure sur les clients **qu'il a vus** (ses 80 %) et sur ceux **qu'il n'a jamais vus** (ses 20 %). Mesure retenue : le critère de l'essai, la **précision moyenne** (aire sous la courbe précision-rappel), et le ROC AUC pour information. Règle de l'essai, fixée à l'avance : écart relatif entre les deux au plus **7 %** ; au-delà, le modèle a appris ses clients par cœur plus que des règles générales.

In [8]:
SEUIL_SURAPPRENTISSAGE = 0.07   # règle de l'essai : écart relatif entraînement - test du critère
lignes = []
for numero, (m, (entrainement, validation)) in enumerate(zip(modeles, plis), start=1):
    p_vus, p_non_vus = m.predict_proba(X.iloc[entrainement])[:, 1], m.predict_proba(X.iloc[validation])[:, 1]
    ap_vus, ap_non_vus = average_precision_score(y[entrainement], p_vus), average_precision_score(y[validation], p_non_vus)
    ecart = (ap_vus - ap_non_vus) / ap_vus
    lignes.append({"Pli mis de côté": numero,
                   "Précision moyenne, clients vus": round(ap_vus, 3), "Précision moyenne, clients non vus": round(ap_non_vus, 3),
                   "Écart relatif (%)": round(ecart * 100, 1),
                   "ROC AUC, clients vus": round(roc_auc_score(y[entrainement], p_vus), 3),
                   "ROC AUC, clients non vus": round(roc_auc_score(y[validation], p_non_vus), 3),
                   "Verdict": "✅ OK" if ecart <= SEUIL_SURAPPRENTISSAGE else "⚠️ surapprentissage"})
display(pd.DataFrame(lignes).set_index("Pli mis de côté"))

,"Précision moyenne, clients vus","Précision moyenne, clients non vus",Écart relatif (%),"ROC AUC, clients vus","ROC AUC, clients non vus",Verdict
Pli mis de côté,,,,,,
1,0.581,0.559,3.8,0.800,0.795,✅ OK
2,0.580,0.546,5.8,0.803,0.779,✅ OK
3,0.582,0.550,5.5,0.802,0.783,✅ OK
4,0.577,0.563,2.4,0.801,0.782,✅ OK
5,0.581,0.562,3.2,0.804,0.780,✅ OK


**Comment lire** : un écart relatif sous 7 % pour chaque modèle (✅) veut dire qu'il fait presque aussi bien sur des clients nouveaux que sur ceux qu'il a appris : ses prédictions de la démonstration sont fiables. Un ⚠️ demanderait de revoir le réglage avant toute démonstration.

## 5. Évaluation sur le test, face à l'entraînement

Chaque modèle prédit ses clients au seuil choisi à l'étape 3 : d'abord ses 80 % d'entraînement (clients vus), puis ses 20 % de test (clients non vus), l'un sous l'autre. Mesures du projet : taux de rappel, précision des défauts prédits, part des clients déclarés en défaut, score décisionnel F2, ROC AUC. Dernière ligne : les 5 tests réunis, soit les 30 000 clients, chacun prédit par un modèle qui ne l'a jamais vu.

In [12]:
proba = np.zeros(len(y))
decision = np.zeros(len(y), dtype=int)

def mesures_test(y_vrai, p, d):
    return {"Rappel (%)": round(recall_score(y_vrai, d) * 100, 1),
            "Précision des défauts prédits (%)": round(precision_score(y_vrai, d, zero_division=0) * 100, 1),
            "Clients déclarés en défaut (%)": round(d.mean() * 100, 1),
            "Score décisionnel F2": round(fbeta_score(y_vrai, d, beta=2), 3), "ROC AUC": round(roc_auc_score(y_vrai, p), 3)}

lignes = []
for numero, (m, (entrainement, validation)) in enumerate(zip(modeles, plis), start=1):
    # Entraînement (clients vus), puis test (clients non vus), au même seuil
    p_vus = m.predict_proba(X.iloc[entrainement])[:, 1]
    d_vus = m.predict(X.iloc[entrainement]).astype(bool)
    lignes.append({"Pli": numero, "Clients": f"entraînement, vus ({len(entrainement)})", **mesures_test(y[entrainement], p_vus, d_vus)})
    proba[validation] = m.predict_proba(X.iloc[validation])[:, 1]
    decision[validation] = m.predict(X.iloc[validation])
    lignes.append({"Pli": numero, "Clients": f"test, non vus ({len(validation)})",
                   **mesures_test(y[validation], proba[validation], decision[validation].astype(bool))})
lignes.append({"Pli": "Les 5 tests réunis", "Clients": f"non vus ({len(y)})", **mesures_test(y, proba, decision.astype(bool))})
display(pd.DataFrame(lignes).set_index(["Pli", "Clients"]))

Rappel (%)  \
Pli                Clients                                 
1                  entraînement, vus (24000)        36.1   
                   test, non vus (6000)             35.4   
2                  entraînement, vus (24000)        39.2   
                   test, non vus (6000)             38.6   
3                  entraînement, vus (24000)        36.6   
                   test, non vus (6000)             35.4   
4                  entraînement, vus (24000)        38.7   
                   test, non vus (6000)             39.6   
5                  entraînement, vus (24000)        38.7   
                   test, non vus (6000)             37.3   
Les 5 tests réunis non vus (30000)                  37.3   

                                              Précision des défauts prédits (%)  \
Pli                Clients                                                        
1                  entraînement, vus (24000)                               69.5   
                   test, non vus (6000)                                    66.6   
2                  entraînement, vus (24000)                               67.1   
                   test, non vus (6000)                                    64.9   
3                  entraînement, vus (24000)                               69.1   
                   test, non vus (6000)                                    67.1   
4                  entraînement, vus (24000)                               66.5   
                   test, non vus (6000)                                    68.3   
5                  entraînement, vus (24000)                               67.8   
                   test, non vus (6000)                                    66.4   
Les 5 tests réunis non vus (30000)                                         66.6   

                                              Clients déclarés en défaut (%)  \
Pli                Clients                                                     
1                  entraînement, vus (24000)                            11.5   
                   test, non vus (6000)                                 11.8   
2                  entraînement, vus (24000)                            12.9   
                   test, non vus (6000)                                 13.2   
3                  entraînement, vus (24000)                            11.7   
                   test, non vus (6000)                                 11.7   
4                  entraînement, vus (24000)                            12.9   
                   test, non vus (6000)                                 12.8   
5                  entraînement, vus (24000)                            12.6   
                   test, non vus (6000)                                 12.4   
Les 5 tests réunis non vus (30000)                                      12.4   

                                              Score décisionnel F2  ROC AUC  
Pli                Clients                                                   
1                  entraînement, vus (24000)                 0.399    0.800  
                   test, non vus (6000)                      0.390    0.795  
2                  entraînement, vus (24000)                 0.428    0.803  
                   test, non vus (6000)                      0.420    0.779  
3                  entraînement, vus (24000)                 0.404    0.802  
                   test, non vus (6000)                      0.391    0.783  
4                  entraînement, vus (24000)                 0.422    0.801  
                   test, non vus (6000)                      0.432    0.782  
5                  entraînement, vus (24000)                 0.423    0.804  
                   test, non vus (6000)                      0.409    0.780  
Les 5 tests réunis non vus (30000)                           0.409    0.784

**Comment lire** : pour chaque pli, la ligne du test doit rester proche de celle de l'entraînement ; un test nettement en dessous voudrait dire que le modèle a appris ses clients par cœur. Des tests proches d'un pli à l'autre veulent dire que les modèles sont stables ; la dernière ligne est ce que la démonstration retrouvera sur tout le dataset. Ce sont les mesures de la section 4 de l'essai, au même seuil.

## 6. Contrôle face à l'essai

Les probabilités et les décisions recalculées doivent être celles enregistrées par l'essai, client par client.

In [13]:
essai = pd.read_csv(root_dir / "data" / "ML" / "comparatif_global" / "jeu_de_base.csv").set_index("ID").loc[original["ID"]]
ecart = np.abs(proba - essai["proba | Complet | CatBoost"].to_numpy()).max()
memes_decisions = (decision == essai["decision | Complet | CatBoost"].to_numpy()).mean()
display(Markdown(f"**Contrôle face à l'essai** : écart maximal des probabilités {ecart:.2e} ; "
                 f"décisions identiques pour {memes_decisions * 100:.2f} % des clients"))
assert ecart < 1e-6 and memes_decisions == 1, "Les modèles ne redonnent pas ceux de l'essai"

**Contrôle face à l'essai** : écart maximal des probabilités 1.11e-16 ; décisions identiques pour 100.00 % des clients

**Comment lire** : un écart de probabilité nul (à l'arrondi de calcul près) et 100 % de décisions identiques veulent dire que les 5 modèles sont exactement ceux de l'essai.

## 7. Enregistrement

In [14]:
ids = original["ID"].to_numpy()
ids_non_vus = [ids[validation].tolist() for _, validation in plis]
assert sorted(sum(ids_non_vus, [])) == sorted(ids.tolist()), "Chaque client doit être non vu par exactement un pli"
fichier = dossier_demo / "demo_3_modeles_plis.joblib"
joblib.dump({"essai": "jeu_de_base", "variante": "Complet", "nom": "CatBoost", "reglage": REGLAGE, "variables": VARIABLES,
             "regle_seuil": f"règle marginale : au plus {BONS_CLIENTS_PAR_DEFAUT} bon client signalé pour 1 défaut, à la marge",
             # Modèle entraîné de chaque pli et son seuil, sans l'enveloppe du choix du seuil (qui garde la fonction gain_net
             # du notebook et ne se recharge pas ailleurs) : un client est déclaré en défaut si sa probabilité atteint le seuil
             "modeles": [m.estimator_ for m in modeles], "seuils": [float(m.best_threshold_) for m in modeles],
             "ids_non_vus": ids_non_vus}, fichier, compress=3)
# Contrôle : le modèle et le seuil enregistrés redonnent les décisions de l'essai
for m, (_, validation) in zip(modeles, plis):
    assert ((m.estimator_.predict_proba(X.iloc[validation])[:, 1] >= m.best_threshold_).astype(int) == decision[validation]).all()
display(Markdown(f"**Enregistré** : `{fichier.name}` ({fichier.stat().st_size / 1e6:.2f} Mo), 5 modèles, "
                 f"{sum(len(i) for i in ids_non_vus)} clients répartis entre eux"))

**Enregistré** : `demo_3_modeles_plis.joblib` (0.85 Mo), 5 modèles, 30000 clients répartis entre eux